# From processed files, move to SILVER level

- The idea is to create the SILVER tables that will be uploaded by data when a certain date in the calendar is marked  as an active operating day.
- The SILVER level includes:
1. Calendar - this is a global level: Day status {W-working, H-holiday, O-operational, C-closed (processed)}.
2. Upload log. We use it to determine how data processing is performed.
3. Payment terminals table. It is a business dimension. We copy it from table terminals_d  (see notebook test data preparation from part 1).
4. Transactions table. It is uploaded from ev2_file_body, operating day in the calendar is "O-operational" and oprational date is equial to transaction date. It is fact  table.
5. Customers table. It is a business dimension. It is uploaded  from the ev2_file_body transaction table by the fields: NAME, PASSPORT. 
6. We create a transaction table from ev2_file_body and put the client's ID  from "Client table" next to each transaction.

As a result of these actions, we will get the following "star" structure:
- Fact table: ev2_silver_transcation;
- Dimansion table: Calendar;
- Dimansion table: Terminals;
- Dimansion table: Customers;

There is also a upload log table to track upload data into silver level.

## 1. Creating a table structure for SILVER LEVEL

## 1.1.  Upload log table **psh_dwh_lh.dbo.ev2_silver_upload_grn**

In [ ]:
print(f"Create Table psh_dwh_lh.dbo.ev2_silver_upload_grn")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_upload_grn")

spark.sql("""
  CREATE TABLE psh_dwh_lh.dbo.ev2_silver_upload_grn (
  upload_id       STRING,
  opdate          DATE,
  idt             TIMESTAMP,
  upload_status   STRING,
  upload_at      TIMESTAMP 
)
USING DELTA;     
"""
)

print(f"Table created")

### 1.2. Payment terminals table **psh_dwh_lh.dbo.ev2_silver_terminals_dim**


In [ ]:
print(f"Create Table psh_dwh_lh.dbo.ev2_silver_terminals_dim")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_terminals_dim")
spark.sql("""
    CREATE TABLE psh_dwh_lh.dbo.ev2_silver_terminals_dim (
            terminal_id   BIGINT,
            terminal_name STRING,
            isactive      STRING,
            upload_id     STRING,
            idt           TIMESTAMP
    )  USING DELTA;  
"""
)

print(f"Table created")

### 1.3. Customers table  **psh_dwh_lh.dbo.ev2_silver_customer_dim**

In [ ]:
print(f"Create Table psh_dwh_lh.dbo.ev2_silver_customers_dim")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_customers_dim")

spark.sql("""
    CREATE TABLE psh_dwh_lh.dbo.ev2_silver_customers_dim (
            cust_id       STRING,
            cust_name     STRING,
            cust_doc      STRING,
            cust_status   STRING,
            upload_id     STRING,
            idt           TIMESTAMP
    )  USING DELTA;  
"""
)

print(f"Table created")

### 1.4. Transactions table  **psh_dwh_lh.dbo.ev2_silver_transactions**

In [ ]:
print(f"psh_dwh_lh.dbo.ev2_silver_transactions")

spark.sql("DROP TABLE IF EXISTS psh_dwh_lh.dbo.ev2_silver_transactions")

spark.sql("""
    CREATE TABLE psh_dwh_lh.dbo.ev2_silver_transactions (
            tx_id         STRING,
            terminal_id   BIGINT,
            opdate        DATE,
            name          STRING,
            passport      STRING,
            amount        DECIMAL(18,2),
            charge        DECIMAL(18,2),
            status        STRING,
            file_name     STRING,
            file_id       STRING,
            upload_id     STRING,
            idt           TIMESTAMP
    )  USING DELTA;  
"""
)

print(f"Table created")

## 2. Сервісні функції для відладки та розробки

### 2.1. Service functions for debugging and development

In [ ]:
# Day statuses
import pandas as pd
df=spark.sql("""
   SELECT distinct A.STATUS
   FROM dbo.calendar A
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

In [ ]:
# select min and max dates from calendar
import pandas as pd
df=spark.sql("""
   SELECT  MIN(A.OPDATE) as midt, MAX(A.OPDATE) as madt 
   FROM dbo.calendar A
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

In [ ]:
# select min and max dates from files content table
import pandas as pd
df=spark.sql("""
   SELECT A.OPDATE, count(*)  as cntrec 
   FROM dbo.ev2_file_body A
   GROUP BY A.OPDATE
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

### Set the status of the operating day in the calendar

In [ ]:
# Set operation day status

pa_dt = "2025-07-12"
# Allowed statuses: W-Working, H-Holiday, O-Open, C-Closed
# Only one operation day can be open
pa_status = "O" 


def set_opdate_status(p_dt, p_status):
    """Set operation day status"""
    # if we set status "O" then all others must be closed in "C"
    if p_status=='O':
        spark.sql("""
                    UPDATE psh_dwh_lh.dbo.calendar
                    SET STATUS='C'
                    WHERE  /*OPDATE = :opdate and*/ STATUS=:status
                """, {"status": p_status})

    # Set the status for the current operation day
    spark.sql("""
                UPDATE psh_dwh_lh.dbo.calendar
                SET STATUS=:status
                WHERE  OPDATE = :opdate
            """, {"opdate": p_dt, "status": p_status})


# run the function 
set_opdate_status(pa_dt, pa_status)
# get the results
df_new_date=spark.sql(
    """
        SELECT C.OPDATE, C.STATUS FROM psh_dwh_lh.dbo.calendar C WHERE C.STATUS='O' 
        UNION ALL
        SELECT C.OPDATE, C.STATUS FROM psh_dwh_lh.dbo.calendar C WHERE C.STATUS='C' 
    """
)
df_new_date.show()



StatementMeta(, 536ba5c0-4693-4bb5-be09-35646bcabfce, 3, Finished, Available, Finished, False)

+----------+------+
|    OPDATE|STATUS|
+----------+------+
|2025-07-12|     O|
|2025-04-15|     C|
|2025-07-02|     C|
+----------+------+

